# データサイエンス実践A 第14講：回帰（単回帰）
### 新潟大学 創生学部（2026年度）

1. 散布図の点の並びに、**いちばん合う直線**（回帰直線）を引く。
1. 直線の**切片・傾き**と**決定係数**を読み取り、言葉で説明する。
1. 回帰の式を使って、**まだ無い値を予測**する。
1. セルは**上から順番に**実行すること（`Shift + Enter` で実行して次のセルへ進む）。

## 目次
**[前回の振り返り](#前回の振り返り)**

**[AO. 回帰（単回帰）](#AO.-回帰（単回帰）)**
- [AO-0 今日の完成形](#AO-0-今日の完成形)
- [AO-1 回帰とは](#AO-1-回帰とは)
- [AO-2 データを読んで散布図を描く](#AO-2-データを読んで散布図を描く)
- [AO-3 回帰直線を求める](#AO-3-回帰直線を求める)
- [AO-4 切片と傾きを読む](#AO-4-切片と傾きを読む)
- [AO-5 決定係数](#AO-5-決定係数)
- [AO-6 回帰直線を描く](#AO-6-回帰直線を描く)
- [AO-7 予測する](#AO-7-予測する)
- [AO-8 回帰の注意点](#AO-8-回帰の注意点)

**[AP. 発展（できる人だけ）](#AP.-発展（できる人だけ）)**
- [AP-1 多項式回帰（曲線を当てはめる）](#AP-1-多項式回帰（曲線を当てはめる）)

**[まとめ](#まとめ)**

---
# 前回の振り返り

- 前回は**関数**（`def` → `return`）と**内包表記**（`[式 for v in リスト]`）を読んだ
- 実データは **Excel で整えてから**（1行目が列名、数値だけ）`pd.read_excel` で読む
- 第４講の**散布図** `plt.scatter(x, y)` で、2つの量の関係を見た。今日はそこに直線を引く

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

from statsmodels.formula.api import ols

- 2行目の `from statsmodels.formula.api import ols` は、回帰の道具 `ols` を使えるようにする行
- `ModuleNotFoundError` が出たら、`statsmodels` が入っていない（Anaconda には最初から入っている）

---
# AO. 回帰（単回帰）

## AO-0 今日の完成形

レストランの食事代とチップの散布図に、**回帰直線**（赤）を引いたもの。直線の式と決定係数をタイトルに入れている。

1. **全部コピーして**自分の notebook のセルに貼り付け、実行する
1. まずは**列名を変えて**、グラフのどこが変わるかを確かめる
1. そのあと部品（[1]〜[4]）の働きを1つずつ学ぶ（AO-1〜AO-6）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from statsmodels.formula.api import ols

df_tip = pd.read_excel('dsa2026data.xlsx', sheet_name='tip_data')

res = ols('tip ~ total_bill', data=df_tip).fit()  # [1] 回帰直線を求める
a = res.params['Intercept']  # [2] 切片
b = res.params['total_bill']  # [2] 傾き
r2 = res.rsquared  # [3] 決定係数

plt.figure(figsize=(8, 5))
plt.scatter(df_tip['total_bill'], df_tip['tip'], alpha=0.5, zorder=2)
plt.plot(df_tip['total_bill'], res.predict(), color='r', zorder=3)  # [4] 回帰直線
plt.title(f'y = {a:.2f} + {b:.3f} x　（決定係数 {r2:.2f}）', fontsize=14)
plt.xlabel('食事代 (USD)', fontsize=14)
plt.ylabel('チップ (USD)', fontsize=14)
plt.show()

### 変えてみよう（1か所変えたら、もう一度実行する）

- `'tip ~ total_bill'` → `'tip ~ size'`（人数とチップ）。`b = res.params['total_bill']` も `'size'` に、`plt.scatter`・`plt.xlabel` も変える
- `color='r'` → `color='orange'`、`alpha=0.5` → `alpha=0.2`

| 番号 | 部品 | 書き方 | 学ぶところ |
|---|---|---|---|
| [1] | 回帰直線を求める | `res = ols('y の列名 ~ x の列名', data=df).fit()` | AO-3 |
| [2] | 切片・傾き | `res.params` | AO-4 |
| [3] | 決定係数 | `res.rsquared` | AO-5 |
| [4] | 回帰直線を描く | `plt.plot(x, res.predict())` | AO-6 |

In [ ]:
# 上のコードを貼り付けて、値を変えてみよう

## AO-1 回帰とは

- 散布図の点の並びに、**いちばん合う直線**を引くこと。直線の式は **y = 切片 + 傾き × x**
  - **切片**：x が 0 のときの y
  - **傾き**：x が 1 増えると、y がどれだけ増えるか
- 「いちばん合う」＝ 点と直線の**縦方向のずれ（の2乗）の合計が最小**になる直線（最小二乗法。`ols` は Ordinary Least Squares の略）
- x が1つのときを**単回帰**、2つ以上のときを**重回帰**（第15講の発展）という

## AO-2 データを読んで散布図を描く

- `tip_data` は、アメリカのレストランの244件の会計のデータ
- 列：`total_bill`（食事代）、`tip`（チップ）、`sex`、`smoker`、`day`（曜日）、`time`（昼・夜）、`size`（人数）

In [ ]:
df_tip = pd.read_excel('dsa2026data.xlsx', sheet_name='tip_data')
df_tip.head()

In [ ]:
plt.scatter(df_tip['total_bill'], df_tip['tip'], alpha=0.5)
plt.xlabel('食事代 (USD)')
plt.ylabel('チップ (USD)')
plt.show()

- 右上がり：食事代が高いほど、チップも多い傾向がある。この傾向を直線で表す

## AO-3 回帰直線を求める

- `ols('y の列名 ~ x の列名', data=データフレーム).fit()` と書く
  - `~` の**左が y**（予測したいもの）、**右が x**（手がかり）。`'tip ~ total_bill'` は「チップを食事代で説明する」
  - `.fit()` で計算する。結果を `res` に入れておく
- 列名に空白や記号があると使えない。Excel の段階で短い列名にしておく（第13講）

In [ ]:
res = ols('tip ~ total_bill', data=df_tip).fit()
print(res.params)

## AO-4 切片と傾きを読む

- `res.params` の `Intercept` が**切片**、`total_bill` が**傾き**
- この例では y = 0.92 + 0.105 × x
  - 傾き 0.105：「食事代が1ドル増えると、チップは約0.1ドル増える」＝ チップはだいたい食事代の1割
  - 切片 0.92：「食事代0ドルなら、チップ0.92ドル」。実際には食事代0の会計は無いので、**切片はあまり意味を持たないことが多い**
- 1つずつ取り出すときは `res.params['Intercept']`、`res.params['total_bill']`

In [ ]:
a = res.params['Intercept']
b = res.params['total_bill']
print('切片', a)
print('傾き', b)

## AO-5 決定係数

- **決定係数**（R²、0〜1）：直線がどれくらい点に合っているか。1 に近いほどよく合う
- 目安：0.5 以上ならまあまあ合っている、0.2 より小さければ直線ではほとんど説明できていない（分野によって目安は違う）

In [ ]:
print('決定係数', res.rsquared)

- 0.46 くらい：チップの多さの半分くらいは食事代で説明できる。残りは人数・曜日・人の気前など、ほかの理由

## AO-6 回帰直線を描く

- `res.predict()` で、データの x それぞれに対する**直線上の y** が求まる。これを `plt.plot` で結ぶと回帰直線になる
- `zorder=` は重なりの順番。大きいほど手前に描かれる（直線を点の手前に）

In [ ]:
plt.scatter(df_tip['total_bill'], df_tip['tip'], alpha=0.5, zorder=2)
plt.plot(df_tip['total_bill'], res.predict(), color='r', zorder=3)
plt.xlabel('食事代 (USD)')
plt.ylabel('チップ (USD)')
plt.show()

### f 文字列で式を書く
- 文字列の前に `f` を付けると、`{変数}` の部分に変数の値が入る。`{a:.2f}` は「小数点以下2けた」

In [ ]:
print(f'y = {a:.2f} + {b:.3f} x')
print(f'決定係数 {res.rsquared:.2f}')

## AO-7 予測する

- 式に x を入れれば、y を予測できる。食事代が 30 ドルなら、チップは 0.92 + 0.105 × 30 ≒ 4.07 ドル
- `res.predict(pd.DataFrame({'total_bill': [30, 50]}))` で、いくつかの x についてまとめて予測できる

In [ ]:
print(a + b * 30)
print(res.predict(pd.DataFrame({'total_bill': [30, 50]})))

- データの範囲（ここでは 3〜51 ドルくらい）の**外側の予測**は当たらないことが多い。「食事代1000ドルならチップ106ドル」とは言えない

### 使ってみよう
- `accident` シートで、`交通事故件数`（x）と `負傷者数`（y）の回帰直線を求め、散布図に描いてみよう
- 傾きを言葉で説明してみよう（「事故が1件増えると、負傷者は○人増える」）

In [ ]:
df_acc = pd.read_excel('dsa2026data.xlsx', sheet_name='accident')

# ここで試してみよう

## AO-8 回帰の注意点

| 注意 | 説明 |
|---|---|
| まず散布図を見る | 点が曲線状に並んでいたら、直線は合わない（AP-1） |
| 関係があっても原因とは限らない | アイスの売上と水難事故は、どちらも「暑さ」で増える。回帰は「一緒に動く」ことしか言えない |
| 外れ値に引っぱられる | 極端な点が1つあると、直線が大きく傾く |
| 範囲の外は予測しない | AO-7 |

---
# AP. 発展（できる人だけ）

## AP-1 多項式回帰（曲線を当てはめる）

- `I(x ** 2)` を足すと2次式（放物線）：y = 切片 + 係数1 × x + 係数2 × x²
- 一人あたりGDPと平均寿命：GDP が低いうちは急に伸び、高くなると伸びが鈍る → 直線より曲線が合う

In [ ]:
df_wb = pd.read_excel('dsa2026data.xlsx', sheet_name='worldbank')
df_wb = df_wb.rename(columns={'一人あたりGDP': 'gdp', '出生時平均寿命': 'life'})

res1 = ols('life ~ gdp', data=df_wb).fit()
res2 = ols('life ~ gdp + I(gdp ** 2)', data=df_wb).fit()
print('直線の決定係数', res1.rsquared)
print('2次式の決定係数', res2.rsquared)

x = np.linspace(df_wb['gdp'].min(), df_wb['gdp'].max(), 100)
plt.scatter(df_wb['gdp'], df_wb['life'])
plt.plot(x, res1.predict(pd.DataFrame({'gdp': x})), color='gray', label='直線')
plt.plot(x, res2.predict(pd.DataFrame({'gdp': x})), color='r', label='2次式')
plt.xlabel('一人あたりGDP (USD)')
plt.ylabel('平均寿命 (歳)')
plt.legend()
plt.show()

- `df.rename(columns={'元の名前': '新しい名前'})` で列名を短くしている
- `np.linspace(最小, 最大, 100)` は、最小から最大までを100等分した数（なめらかな曲線を描くため）
- 次数を上げすぎると、点にぴったり合っても予測には役に立たない曲線になる（過学習）

---
# まとめ

| やりたいこと | 書き方 |
|---|---|
| 準備 | `from statsmodels.formula.api import ols` |
| 回帰直線を求める | `res = ols('y の列名 ~ x の列名', data=df).fit()` |
| 切片・傾き | `res.params`（`res.params['Intercept']`、`res.params['x の列名']`） |
| 決定係数 | `res.rsquared` |
| 回帰直線を描く | `plt.plot(df['x の列名'], res.predict(), color='r')` |
| 予測 | `res.predict(pd.DataFrame({'x の列名': [値]}))` |

- 傾きは「x が 1 増えると y がどれだけ増えるか」。決定係数は直線の合い具合（0〜1）
- 回帰は「一緒に動く」ことしか言えない。原因かどうかは別に考える

次回（第15講）は**時系列**と、**自分で選んだデータでグラフ＋回帰**（第３回課題）。